# 05: Data Types, Bins & Moving Windows (Exercises 61–75)

Explore array subclassing, unbuffered updates with np.add.at, bincount weighted accumulations, and moving averages.

---


In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")

### Exercise 61: Find the nearest value from a given value in an array
**Difficulty:** `★★☆`  
**Tags:** `Searching, Flat Indices`

#### 💡 Intuition & Concept
`np.abs(Z - target).argmin()` finds the 1D flat index of the element closest to target. Slicing with `Z.flat[...]` retrieves the scalar value regardless of array dimensionality.

#### ⚠️ Key Takeaway & Gotchas
`Z.flat` is an iterator that handles multi-dimensional arrays without needing explicit reshaping.


In [ ]:
rng = np.random.default_rng(seed=42)
Z = rng.uniform(0, 1, 10)
target = 0.5
nearest_val = Z.flat[np.abs(Z - target).argmin()]
print("Array:", np.round(Z, 3))
print(f"Target: {target} -> Nearest: {nearest_val:.3f}")

### Exercise 62: Considering two arrays with shape (1,3) and (3,1), compute their sum using an iterator
**Difficulty:** `★★☆`  
**Tags:** `Iteration, nditer`

#### 💡 Intuition & Concept
`np.nditer([A, B, None])` allows iterating over multiple arrays simultaneously with automatic broadcasting.

#### ⚠️ Key Takeaway & Gotchas
The `None` operand creates an output buffer array matching the broadcasted shape `(3, 3)`.


In [ ]:
A = np.arange(3).reshape(3, 1)
B = np.arange(3).reshape(1, 3)
it = np.nditer([A, B, None])
for x, y, z in it:
    z[...] = x + y
res = it.operands[2]
print("Result of nditer broadcasted sum:\n", res)

### Exercise 63: Create an array class that has a name attribute
**Difficulty:** `★★☆`  
**Tags:** `OOP, Subclassing`

#### 💡 Intuition & Concept
Subclassing `np.ndarray` requires overriding `__new__` for allocation and `__array_finalize__` so that slices and views retain custom attributes.

#### ⚠️ Key Takeaway & Gotchas
Failing to implement `__array_finalize__` causes attributes to be lost when slicing or copying.


In [ ]:
class NamedArray(np.ndarray):
    def __new__(cls, array, name="anonymous"):
        obj = np.asarray(array).view(cls)
        obj.name = name
        return obj
        
    def __array_finalize__(self, obj):
        if obj is None: return
        self.name = getattr(obj, 'name', "anonymous")

a = NamedArray([1, 2, 3], name="MySensors")
print("Array:", a)
print("Name:", a.name)
# Slices preserve the attribute:
sliced = a[:2]
print("Slice name:", sliced.name)

### Exercise 64: Add 1 to elements indexed by another vector (handling repeated indices)
**Difficulty:** `★★☆`  
**Tags:** `Indexing, np.add.at`

#### 💡 Intuition & Concept
Standard assignment `Z[I] += 1` executes buffered assignments where repeated indices overwrite each other instead of accumulating. `np.add.at(Z, I, 1)` performs unbuffered in-place accumulation.

#### ⚠️ Key Takeaway & Gotchas
For histogram counting, `np.bincount` is often even faster if you just want frequencies.


In [ ]:
Z = np.zeros(10)
I = np.array([1, 2, 2, 2, 4, 5, 5])
# Using unbuffered ufunc method:
np.add.at(Z, I, 1)
print("Result with np.add.at:", Z)

### Exercise 65: Accumulate elements of a vector (X) to an array (F) based on an index list (I)
**Difficulty:** `★★☆`  
**Tags:** `Histograms, np.bincount`

#### 💡 Intuition & Concept
`np.bincount(I, weights=X)` sums the weights `X` into bins indexed by `I`, achieving fast group accumulation.

#### ⚠️ Key Takeaway & Gotchas
Negative values in `I` are not supported by `bincount`.


In [ ]:
X = [1, 2, 3, 4, 5, 6]
I = [1, 3, 9, 3, 4, 1]
F = np.bincount(I, weights=X)
print("Bincount accumulated weights:\n", F)

### Exercise 66: Considering a (w,h,3) image of ubyte, compute the number of unique colors
**Difficulty:** `★★☆`  
**Tags:** `Images, Unique`

#### 💡 Intuition & Concept
Reshape the image array to `(-1, 3)` so each pixel is a row, then pass `axis=0` to `np.unique` to find unique RGB color triplets.

#### ⚠️ Key Takeaway & Gotchas
An alternative trick is packing 3 ubytes into a single 24-bit / 32-bit integer for ultra-fast 1D unique extraction.


In [ ]:
w, h = 16, 16
rng = np.random.default_rng(seed=42)
img = rng.integers(0, 4, (h, w, 3), dtype=np.ubyte)

# Method 1: 2D unique rows
unique_colors = np.unique(img.reshape(-1, 3), axis=0)
print("Unique colors count:", len(unique_colors))

### Exercise 67: Considering a 4D array, how to get sum over the last two axis at once?
**Difficulty:** `★★☆`  
**Tags:** `Reductions, Multi-Axis`

#### 💡 Intuition & Concept
The `axis` argument of `sum` accepts a tuple of dimensions. Passing `axis=(-2, -1)` sums across the last two axes simultaneously.

#### ⚠️ Key Takeaway & Gotchas
In older Python/NumPy code people would reshape before summing, but tuple axis is faster and cleaner.


In [ ]:
A = np.ones((2, 3, 4, 5))
sum_last_two = A.sum(axis=(-2, -1))
print("Original shape: ", A.shape)
print("Reduced shape:  ", sum_last_two.shape)
print("Values (4x5=20):", sum_last_two[0, 0])

### Exercise 68: Compute means of subsets of D using vector S of same size describing subset indices
**Difficulty:** `★★☆`  
**Tags:** `Groupby, Bincount`

#### 💡 Intuition & Concept
Compute subset sums via `np.bincount(S, weights=D)` and counts via `np.bincount(S)`. Dividing sums by counts yields vectorized subset means without Python loops.

#### ⚠️ Key Takeaway & Gotchas
If a subset index has 0 occurrences, division by zero results in `nan`.


In [ ]:
rng = np.random.default_rng(seed=42)
D = rng.uniform(0, 10, 20)
S = rng.integers(0, 3, 20)

sums = np.bincount(S, weights=D)
counts = np.bincount(S)
means = sums / counts
for group, mean in enumerate(means):
    print(f"Group {group}: count={counts[group]}, mean={mean:.3f}")

### Exercise 69: How to get the diagonal of a dot product efficiently?
**Difficulty:** `★★☆`  
**Tags:** `Performance, Linear Algebra`

#### 💡 Intuition & Concept
Computing `np.diag(A @ B)` calculates the entire $O(N^3)$ matrix product and discards $(N^2 - N)$ elements. Computing `np.sum(A * B.T, axis=1)` calculates only the diagonal elements in $O(N^2)$ time.

#### ⚠️ Key Takeaway & Gotchas
This saves massive memory and compute time for large matrices.


In [ ]:
A = np.random.default_rng(42).random((100, 100))
B = np.random.default_rng(42).random((100, 100))

# Slow: np.diag(A @ B) -> O(N^3)
# Fast:
diag_fast = np.sum(A * B.T, axis=1)
diag_einsum = np.einsum("ij,ji->i", A, B)
print("Matches?", np.allclose(diag_fast, diag_einsum))

### Exercise 70: Consider vector [1,2,3,4,5], build new vector with 3 consecutive zeros interleaved
**Difficulty:** `★★☆`  
**Tags:** `Strided Slicing, Interleaving`

#### 💡 Intuition & Concept
Allocate an array of zeros with length $L = len(Z) + (len(Z) - 1) \times nz$, and assign values using step slice `::nz + 1`.

#### ⚠️ Key Takeaway & Gotchas
The step stride neatly places each original value with exactly 3 zeros between them.


In [ ]:
Z = np.array([1, 2, 3, 4, 5])
nz = 3
Z0 = np.zeros(len(Z) + (len(Z) - 1) * nz, dtype=Z.dtype)
Z0[::nz + 1] = Z
print(Z0)

### Exercise 71: Multiply an array of dimension (5,5,3) by an array with dimensions (5,5)
**Difficulty:** `★★☆`  
**Tags:** `Broadcasting, Dimensions`

#### 💡 Intuition & Concept
Add a trailing dimension to the 2D array: `B[:, :, None]` gives shape `(5, 5, 1)`, which broadcasts across the 3 channels of `(5, 5, 3)`.

#### ⚠️ Key Takeaway & Gotchas
Element-wise multiplication `A * B` fails directly because dimensions align from right to left.


In [ ]:
A = np.ones((5, 5, 3))
B = 2 * np.ones((5, 5))
C = A * B[:, :, None]
print("Result shape:", C.shape)
print("Value at (0,0):", C[0, 0])

### Exercise 72: How to swap two rows of an array?
**Difficulty:** `★★☆`  
**Tags:** `Indexing, In-place`

#### 💡 Intuition & Concept
Fancy indexing `A[[0, 1]] = A[[1, 0]]` swaps rows 0 and 1 cleanly.

#### ⚠️ Key Takeaway & Gotchas
The right-hand side creates a temporary copy before assignment, ensuring data integrity during swap.


In [ ]:
A = np.arange(16).reshape(4, 4)
print("Original:\n", A)
A[[0, 1]] = A[[1, 0]]
print("After swapping rows 0 & 1:\n", A)

### Exercise 73: Find unique line segments composing 10 triangles (with shared vertices)
**Difficulty:** `★★★`  
**Tags:** `Geometry, Unique Edges`

#### 💡 Intuition & Concept
Triangles are defined by vertex indices $(v_0, v_1, v_2)$. Extracting edges $(v_0,v_1), (v_1,v_2), (v_2,v_0)$, sorting endpoints so $v_i \le v_j$, and deduplicating reveals all unique mesh edges.

#### ⚠️ Key Takeaway & Gotchas
Sorting the endpoints of each edge ensures that undirected edge $(A, B)$ matches $(B, A)$.


In [ ]:
faces = np.random.default_rng(42).integers(0, 10, (5, 3))
edges = np.vstack([faces[:, [0, 1]], faces[:, [1, 2]], faces[:, [2, 0]]])
edges = np.sort(edges, axis=1)
unique_edges = np.unique(edges, axis=0)
print(f"Total triangle edges: {len(edges)}")
print(f"Unique wireframe edges: {len(unique_edges)}")
print(unique_edges)

### Exercise 74: Given a sorted array C that corresponds to a bincount, produce array A such that np.bincount(A) == C
**Difficulty:** `★★☆`  
**Tags:** `np.repeat, Inversion`

#### 💡 Intuition & Concept
`np.repeat(np.arange(len(C)), C)` repeats integer $i$ exactly $C[i]$ times, perfectly inverting `bincount`.

#### ⚠️ Key Takeaway & Gotchas
Extremely concise and $O(N)$ fast.


In [ ]:
C = np.array([0, 2, 3, 1, 0, 4])
A = np.repeat(np.arange(len(C)), C)
print("Target counts C:   ", C)
print("Reconstructed A:   ", A)
print("Verification bincount(A):", np.bincount(A, minlength=len(C)))
assert np.array_equal(np.bincount(A, minlength=len(C)), C)

### Exercise 75: How to compute averages using a sliding window over an array?
**Difficulty:** `★★★`  
**Tags:** `Moving Average, Cumsum`

#### 💡 Intuition & Concept
A moving average of window size $k$ can be computed using cumulative sums: subtract `cumsum[i - k]` from `cumsum[i]` in $O(N)$ time.

#### ⚠️ Key Takeaway & Gotchas
`np.convolve(arr, np.ones(k)/k, mode='valid')` is another clean alternative.


In [ ]:
def moving_average(a, n=3):
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

Z = np.arange(10)
print("Array:          ", Z)
print("Moving avg (w=3):", moving_average(Z, n=3))